# SVM RBF — Two-Stage Pipeline Grid Search (Colab)
**Complete workflow in one notebook:**
1. **Stage 1 Grid Search** — Binary SVM (Background vs Wetland), C × gamma → 4 runs
2. **Stage 2 Grid Search** — Multi-class SVM (Wetland types 1–5), C × gamma → 4 runs
3. **Full Pipeline Evaluation** — Best Stage 1 + Best Stage 2, end-to-end stats

**Truth-source class mapping:**
- 0 = Background
- 1 = Fen (Graminoid) | 2 = Fen (Woody) | 3 = Marsh | 4 = Shallow Open Water | 5 = Swamp

**Requirements:** Colab GPU runtime (T4 or better) for cuML. All models + stats saved to `MyDrive/SVM_Pipeline/`.

In [ ]:
# ── Cell 1: Mount Google Drive & set save directory ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

SAVE_DIR = '/content/drive/MyDrive/SVM_Pipeline'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✔ Save directory: {SAVE_DIR}")

In [ ]:
# ── Cell 2: Upload dataset locally to Colab instance ─────────────────────────
# Upload wetland_dataset_middle_split.npz from your machine.
# Keeping it local (/content/) is much faster for I/O than reading from Drive.
from google.colab import files

print("Select wetland_dataset_middle_split.npz to upload...")
uploaded = files.upload()

DATA_PATH = '/content/wetland_dataset_middle_split.npz'
print(f"\n✔ Dataset available at: {DATA_PATH}")

In [ ]:
# ── Cell 3: Check cuML backend ────────────────────────────────────────────────
try:
    from cuml.svm import SVC
    BACKEND  = "cuML (GPU)"
    USE_CUML = True
    print(f"✔ {BACKEND} available")
except ImportError:
    from sklearn.svm import SVC
    BACKEND  = "sklearn (CPU)"
    USE_CUML = False
    print(f"⚠ cuML not found — falling back to {BACKEND}")
    print("  NOTE: sklearn SVC on 1.5M samples will be extremely slow.")
    print("  Make sure you selected a GPU runtime (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# ── Cell 4: Imports + Hyperparameters ────────────────────────────────────────
import numpy as np
import json
import joblib
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import StandardScaler

# Class name lookup (truth-source mapping)
CLASS_NAMES = {
    0: "Background",
    1: "Fen (Graminoid)",
    2: "Fen (Woody)",
    3: "Marsh",
    4: "Shallow Open Water",
    5: "Swamp",
}

# ── Fixed hyperparameters ─────────────────────────────────────────────────────
# C=1.0  : moderate regularization, well-suited for 1.5M samples (faster,
#           avoids overfitting without sacrificing boundary quality)
# gamma='scale' : auto-computes 1/(n_features * var(X)) ≈ 1/64 after
#           StandardScaling — data-adaptive and far better than a fixed small
#           value like 0.001 which would underfit with 64 features
SVC_C     = 1.0
SVC_GAMMA = "scale"

print(f"Hyperparameters: C={SVC_C}  gamma={SVC_GAMMA}")
print("Imports ready.")


In [ ]:
# ── Cell 5: Load data, scale, and prepare binary + wetland labels ─────────────
data = np.load(DATA_PATH)
X_train_raw  = data['X_train']
y_train_raw  = data['y_train']
X_test_raw   = data['X_test']
y_test_raw   = data['y_test']
test_row_min = int(data['test_row_min'])
test_row_max = int(data['test_row_max'])
data.close()

print(f"Train: {X_train_raw.shape[0]:,}  |  Test: {X_test_raw.shape[0]:,}  |  Features: {X_train_raw.shape[1]}")

# Fit scaler on training data only — this scaler is reused for Stage 2
print("\nFitting StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw.astype(np.float32))
X_test_scaled  = scaler.transform(X_test_raw.astype(np.float32))

SCALER_PATH = os.path.join(SAVE_DIR, 'svm_rbf_scaler.pkl')
joblib.dump(scaler, SCALER_PATH)
print(f"Scaler saved: {SCALER_PATH}")

# Stage 1 binary labels (0=Background, 1=Wetland)
y_train_binary = (y_train_raw != 0).astype(np.int32)
y_test_binary  = (y_test_raw  != 0).astype(np.int32)

# Stage 2 wetland-only data (classes 1–5, background filtered)
s2_train_mask  = y_train_raw != 0
X_train_s2     = X_train_scaled[s2_train_mask]
y_train_s2     = y_train_raw[s2_train_mask].astype(np.int32)
s2_test_mask   = y_test_raw != 0
X_test_s2      = X_test_scaled[s2_test_mask]
y_test_s2      = y_test_raw[s2_test_mask].astype(np.int32)

print(f"\nStage 1 — Train: {X_train_scaled.shape[0]:,}  |  Test: {X_test_scaled.shape[0]:,}")
print(f"Stage 2 — Train: {X_train_s2.shape[0]:,}  |  Test: {X_test_s2.shape[0]:,}")

# Per-class weights for Stage 2 (balanced + Fen Graminoid dampened)
CLASS1_DAMPEN = 0.4
s2_unique, s2_counts = np.unique(y_train_s2, return_counts=True)
s2_weights = {int(c): float(len(y_train_s2) / (len(s2_unique) * n)) for c, n in zip(s2_unique, s2_counts)}
s2_weights[1] = s2_weights[1] * CLASS1_DAMPEN
print(f"\nStage 2 class weights (Fen Graminoid dampened x{CLASS1_DAMPEN}):")
for cls, w in s2_weights.items():
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {w:.4f}")

## Stage 1 — Binary SVM (Background vs Wetland)
Trains a single RBF SVM to separate background (class 0) from all wetland pixels (classes 1–5). Uses `class_weight='balanced'` to handle the class imbalance automatically.

In [ ]:
# ── Cell 6: Stage 1 — Train Binary SVM (Background vs Wetland) ───────────────
import time, pickle
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("=" * 68)
print(f"  STAGE 1  —  Binary SVM  |  C={SVC_C}  gamma={SVC_GAMMA}  [{BACKEND}]")
print("=" * 68)

if USE_CUML:
    from cuml.svm import SVC as cuSVC
    s1_clf = cuSVC(C=SVC_C, gamma=SVC_GAMMA, kernel="rbf",
                   class_weight="balanced", probability=False)
else:
    from sklearn.svm import SVC as skSVC
    s1_clf = skSVC(C=SVC_C, gamma=SVC_GAMMA, kernel="rbf",
                   class_weight="balanced", probability=False)

print("\nTraining …", flush=True)
t0 = time.time()
s1_clf.fit(X_train_scaled, y_train_binary)
t_train = time.time() - t0
print(f"Train time : {t_train/60:.1f} min  ({t_train:.0f} s)")

print("Evaluating …", flush=True)
t1 = time.time()
y_pred_s1 = s1_clf.predict(X_test_scaled)
if USE_CUML:
    y_pred_s1 = np.array(y_pred_s1.to_numpy(), dtype=int)
else:
    y_pred_s1 = np.array(y_pred_s1, dtype=int)
t_eval = time.time() - t1

# Metrics
s1_acc = accuracy_score(y_test_binary, y_pred_s1)
report = classification_report(
    y_test_binary, y_pred_s1,
    target_names=["Background", "Wetland"],
    output_dict=True, zero_division=0
)
f1_bg  = report["Background"]["f1-score"]
f1_wet = report["Wetland"]["f1-score"]

print(f"\n  Accuracy      : {s1_acc*100:.2f}%")
print(f"\n  {'Class':<14} {'Precision':>9} {'Recall':>9} {'F1':>9} {'Support':>9}")
print(f"  {'─'*52}")
for name in ["Background", "Wetland"]:
    r   = report[name]
    sup = int(r["support"])
    print(f"  {name:<14} {r['precision']:>9.4f} {r['recall']:>9.4f} {r['f1-score']:>9.4f} {sup:>9,}")

cm = confusion_matrix(y_test_binary, y_pred_s1)
print(f"\n  Confusion matrix:")
print(f"                    Pred BG   Pred Wet")
print(f"  True Background {cm[0,0]:>9,} {cm[0,1]:>9,}")
print(f"  True Wetland    {cm[1,0]:>9,} {cm[1,1]:>9,}")
print(f"\n  Eval time  : {t_eval:.1f} s")

# Save model to Drive
ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
s1_file = f"{SAVE_DIR}/svm_rbf_s1_C{SVC_C}_gamma{SVC_GAMMA}_{ts}.pkl"
with open(s1_file, "wb") as f:
    pickle.dump(s1_clf, f)
print(f"\n  Model saved: {s1_file}")

# Store metadata + aliases used by final evaluation cell
s1_meta = dict(C=SVC_C, gamma=str(SVC_GAMMA),
               accuracy=round(s1_acc, 6),
               f1_background=round(f1_bg, 6),
               f1_wetland=round(f1_wet, 6),
               model_file=s1_file,
               train_time_s=round(t_train, 1))
best_s1_model = s1_clf
best_s1_meta  = s1_meta
best_s1_file  = s1_file

print("\n  Stage 1 complete.")


## Stage 2 — Wetland Multi-Class SVM (Classes 1–5)
Trains a single RBF SVM on wetland-only pixels to distinguish the 5 wetland types. Reuses the Stage 1 scaler — **not refitted**. Fen (Graminoid) weight is dampened ×0.4 to prevent it from dominating the decision boundary.

In [ ]:
# ── Cell 7: Stage 2 — Train Wetland Multi-Class SVM (Classes 1–5) ────────────
import time, pickle
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

WETLAND_CLASSES = [1, 2, 3, 4, 5]
WETLAND_NAMES   = {1: "Fen (Graminoid)", 2: "Fen (Woody)",
                   3: "Marsh",           4: "Shallow OW", 5: "Swamp"}

print("=" * 68)
print(f"  STAGE 2  —  Wetland Multi-Class SVM  |  C={SVC_C}  gamma={SVC_GAMMA}  [{BACKEND}]")
print("=" * 68)
print(f"\n  Train pixels: {X_train_s2.shape[0]:,}  |  Test pixels: {X_test_s2.shape[0]:,}")

if USE_CUML:
    from cuml.svm import SVC as cuSVC
    s2_clf = cuSVC(C=SVC_C, gamma=SVC_GAMMA, kernel="rbf",
                   class_weight=s2_weights, probability=False)
else:
    from sklearn.svm import SVC as skSVC
    s2_clf = skSVC(C=SVC_C, gamma=SVC_GAMMA, kernel="rbf",
                   class_weight=s2_weights, probability=False)

print("\nTraining …", flush=True)
t0 = time.time()
s2_clf.fit(X_train_s2, y_train_s2)
t_train = time.time() - t0
print(f"Train time : {t_train/60:.1f} min  ({t_train:.0f} s)")

print("Evaluating …", flush=True)
t1 = time.time()
y_pred_s2 = s2_clf.predict(X_test_s2)
if USE_CUML:
    y_pred_s2 = np.array(y_pred_s2.to_numpy(), dtype=int)
else:
    y_pred_s2 = np.array(y_pred_s2, dtype=int)
t_eval = time.time() - t1

# Metrics
s2_acc = accuracy_score(y_test_s2, y_pred_s2)
report = classification_report(
    y_test_s2, y_pred_s2,
    labels=WETLAND_CLASSES,
    target_names=[WETLAND_NAMES[c] for c in WETLAND_CLASSES],
    output_dict=True, zero_division=0
)
f1_per_class    = [report[WETLAND_NAMES[c]]["f1-score"] for c in WETLAND_CLASSES]
mean_wetland_f1 = float(np.mean(f1_per_class))

print(f"\n  Accuracy (wetland pixels) : {s2_acc*100:.2f}%")
print(f"  Mean Wetland F1           : {mean_wetland_f1:.4f}")
print(f"\n  {'Class':<22} {'Precision':>9} {'Recall':>9} {'F1':>9} {'Support':>9}")
print(f"  {'─'*60}")
for c in WETLAND_CLASSES:
    nm  = WETLAND_NAMES[c]
    r   = report[nm]
    sup = int(r["support"])
    print(f"  {nm:<22} {r['precision']:>9.4f} {r['recall']:>9.4f} {r['f1-score']:>9.4f} {sup:>9,}")

cm = confusion_matrix(y_test_s2, y_pred_s2, labels=WETLAND_CLASSES)
print(f"\n  Confusion matrix (rows=true, cols=pred):")
print("  " + " " * 22 + "".join(f"{c:>8}" for c in WETLAND_CLASSES))
for i, c in enumerate(WETLAND_CLASSES):
    print("  " + f"{WETLAND_NAMES[c]:<22}" +
          "".join(f"{cm[i,j]:>8}" for j in range(len(WETLAND_CLASSES))))
print(f"\n  Eval time  : {t_eval:.1f} s")

# Save model to Drive
ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
s2_file = f"{SAVE_DIR}/svm_rbf_s2_C{SVC_C}_gamma{SVC_GAMMA}_{ts}.pkl"
with open(s2_file, "wb") as f:
    pickle.dump(s2_clf, f)
print(f"\n  Model saved: {s2_file}")

# Store metadata + aliases used by final evaluation cell
s2_meta = dict(C=SVC_C, gamma=str(SVC_GAMMA),
               accuracy=round(s2_acc, 6),
               mean_wetland_f1=round(mean_wetland_f1, 6),
               per_class_f1={WETLAND_NAMES[c]: round(report[WETLAND_NAMES[c]]["f1-score"], 6)
                             for c in WETLAND_CLASSES},
               model_file=s2_file,
               train_time_s=round(t_train, 1))
best_s2_model = s2_clf
best_s2_meta  = s2_meta
best_s2_file  = s2_file

print("\n  Stage 2 complete.")


## Final Pipeline Evaluation — Best Stage 1 + Best Stage 2

Applies `best_s1_model` to the full test set to predict wetland vs background, then routes wetland pixels through `best_s2_model` to get fine-grained class labels 1–5. Evaluates the combined result against ground truth for all 6 classes (0–5).

In [ ]:
# ── Cell 8: Final Pipeline Evaluation ────────────────────────────────────────
import json, time
from datetime import datetime
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)
import numpy as np

ALL_CLASSES = [0, 1, 2, 3, 4, 5]
ALL_NAMES   = {0: "Background",      1: "Fen (Graminoid)",
               2: "Fen (Woody)",     3: "Marsh",
               4: "Shallow OW",      5: "Swamp"}
WETLAND_CLASSES = [1, 2, 3, 4, 5]

print("=" * 72)
print("  FINAL PIPELINE EVALUATION")
print(f"  Stage 1 best: C={best_s1_meta['C']}  gamma={best_s1_meta['gamma']}")
print(f"              → binary F1 (wetland): {best_s1_meta['f1_wetland']:.4f}")
print(f"  Stage 2 best: C={best_s2_meta['C']}  gamma={best_s2_meta['gamma']}")
print(f"              → mean wetland F1   : {best_s2_meta['mean_wetland_f1']:.4f}")
print("=" * 72)

# ── Stage 1: classify all test pixels ────────────────────────────────────────
print("\n[1/3] Running Stage 1 on full test set …", flush=True)
t0 = time.time()
s1_pred = best_s1_model.predict(X_test_scaled)
if USE_CUML:
    s1_pred = np.array(s1_pred.to_numpy(), dtype=int)
else:
    s1_pred = np.array(s1_pred, dtype=int)
print(f"      Done in {time.time()-t0:.1f} s")

wetland_mask  = s1_pred == 1                       # predicted wetland pixels
n_total       = len(s1_pred)
n_wet_pred    = int(wetland_mask.sum())
n_bg_pred     = n_total - n_wet_pred
print(f"      Total pixels : {n_total:,}")
print(f"      → Background : {n_bg_pred:,}  ({n_bg_pred/n_total*100:.1f}%)")
print(f"      → Wetland    : {n_wet_pred:,}  ({n_wet_pred/n_total*100:.1f}%)")

# ── Stage 2: classify wetland pixels ─────────────────────────────────────────
print("\n[2/3] Running Stage 2 on wetland pixels …", flush=True)
t0 = time.time()
X_wet = X_test_scaled[wetland_mask]
s2_pred_wet = best_s2_model.predict(X_wet)
if USE_CUML:
    s2_pred_wet = np.array(s2_pred_wet.to_numpy(), dtype=int)
else:
    s2_pred_wet = np.array(s2_pred_wet, dtype=int)
print(f"      Done in {time.time()-t0:.1f} s")

# ── Merge into final prediction array ────────────────────────────────────────
final_pred = np.zeros(n_total, dtype=int)           # default = background (0)
final_pred[wetland_mask] = s2_pred_wet

# ── Evaluate against ground truth ────────────────────────────────────────────
print("\n[3/3] Computing metrics …", flush=True)
y_true = y_test_raw                                 # all 1,894,906 test labels (0–5)

pipeline_acc     = accuracy_score(y_true, final_pred)
pipeline_wf1     = f1_score(y_true, final_pred, average="weighted", zero_division=0)
pipeline_mean_w  = float(np.mean([
    f1_score(y_true == c, final_pred == c, average="binary", zero_division=0)
    for c in WETLAND_CLASSES
]))

report = classification_report(
    y_true, final_pred,
    labels=ALL_CLASSES,
    target_names=[ALL_NAMES[c] for c in ALL_CLASSES],
    output_dict=True, zero_division=0
)

# ── Print final summary ───────────────────────────────────────────────────────
print(f"\n{'='*72}")
print("  COMBINED PIPELINE RESULTS")
print(f"{'='*72}")
print(f"  Overall Accuracy    : {pipeline_acc*100:.2f}%")
print(f"  Weighted F1         : {pipeline_wf1:.4f}")
print(f"  Mean Wetland F1     : {pipeline_mean_w:.4f}")
print()

hdr = f"  {'Class':<22} {'Precision':>9} {'Recall':>9} {'F1':>9} {'Support':>9}"
print(hdr)
print(f"  {'─'*58}")
for c in ALL_CLASSES:
    nm  = ALL_NAMES[c]
    r   = report[nm]
    sup = int(r["support"])
    print(f"  {nm:<22} {r['precision']:>9.4f} {r['recall']:>9.4f} "
          f"{r['f1-score']:>9.4f} {sup:>9,}")
print(f"  {'─'*58}")
print(f"  {'Macro avg':<22} {'':>9} {'':>9} {report['macro avg']['f1-score']:>9.4f}")
print(f"  {'Weighted avg':<22} {'':>9} {'':>9} {pipeline_wf1:>9.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, final_pred, labels=ALL_CLASSES)
print(f"\n  Confusion matrix (rows=true, cols=pred):")
print("  " + " " * 22 + "".join(f"{c:>8}" for c in ALL_CLASSES))
for i, c in enumerate(ALL_CLASSES):
    print("  " + f"{ALL_NAMES[c]:<22}" + "".join(f"{cm[i,j]:>8}" for j in range(len(ALL_CLASSES))))

# ── Save stats JSON ───────────────────────────────────────────────────────────
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
stats = {
    "model":             "SVM RBF Two-Stage Pipeline",
    "timestamp":         ts,
    "backend":           BACKEND,
    "stage1_params":     {"C": best_s1_meta["C"], "gamma": best_s1_meta["gamma"]},
    "stage2_params":     {"C": best_s2_meta["C"], "gamma": best_s2_meta["gamma"]},
    "stage1_model_file": best_s1_file,
    "stage2_model_file": best_s2_file,
    "accuracy":          round(pipeline_acc, 6),
    "weighted_f1":       round(pipeline_wf1, 6),
    "mean_wetland_f1":   round(pipeline_mean_w, 6),
    "per_class": {
        ALL_NAMES[c]: {
            "precision": round(report[ALL_NAMES[c]]["precision"], 6),
            "recall":    round(report[ALL_NAMES[c]]["recall"], 6),
            "f1":        round(report[ALL_NAMES[c]]["f1-score"], 6),
            "support":   int(report[ALL_NAMES[c]]["support"])
        } for c in ALL_CLASSES
    },
    "confusion_matrix": cm.tolist()
}

# Save to Google Drive
drive_path = f"{SAVE_DIR}/svm_combo_pipeline_stats_{ts}.json"
with open(drive_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"\n  Stats saved to Drive : {drive_path}")

# Also save to Statistics/SVM/ folder for consistency with other models
import os
stats_dir = "/content/drive/MyDrive/Wetland-Mapping-ELEC498-Group-46/Statistics/SVM"
os.makedirs(stats_dir, exist_ok=True)
stats_path = f"{stats_dir}/Model_SVM_RBF_Pipeline_statistics_{ts}.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"  Stats saved to repo  : {stats_path}")

print(f"\n{'='*72}")
print("  PIPELINE COMPLETE")
print(f"{'='*72}")
